In [ ]:
# ============================================================
# notebooks/08_fusion_model_training.py
# Run after 07_contextual_features.py:
#   python notebooks/08_fusion_model_training.py
#
# PURPOSE: Two-stage notebook:
#   Stage 1 — Generate real Phase 1 risk scores for all 590K rows
#             using the full EARN+ pipeline (AE + ResNet → 128 →
#             Nystroem → IPCA → AttentionRXTJ + IFM ensemble).
#             Patches contextual_features.npy column 7 in-place.
#   Stage 2 — Train FusionNet v2 (8→64→32→16→1 with feature attention)
#             using Jaya threshold optimisation. Saves fusion_net.pt,
#             fusion_scaler.pkl, fusion_config.json, ROC plots.
#
# WHAT WAS FIXED (merged from fix_retrain_nb08.py):
#   - AE architecture: added Dropout at indices [3] and [7] and removed
#     the spurious extra ReLU after the final encoder Linear — these exact
#     positions match the autoencoder.pt state_dict keys (encoder.0,1,4,5,8)
#   - Pipeline order: raw (224) → AE(64) + ResNet(64) → concat(128) →
#     Nystroem(128→300) → IPCA(300→50) — NOT raw(224) → Nystroem directly
#   - AttentionRXTJ: uses exact app.py names (paths ModuleList, attn_gru,
#     attention Sequential, classifier Sequential) so load_state_dict works
#   - FusionNet v2: wider (8→64→32→16→1) + CosineAnnealingLR (removes the
#     deprecated verbose=True ReduceLROnPlateau) + monitors AUC not loss
#   - Training: EPOCHS=200, PATIENCE=25, batch=4096, lr=5e-4
#   - Smart resume: skips Stage 1 if model_probs_full.npy already exists
#     with the correct row count
#
# OUTPUTS:
#   models/fusion_net.pt          — trained FusionNet v2 weights
#   models/fusion_scaler.pkl      — StandardScaler for 8 input features
#   results/fusion_config.json    — thresholds + metrics + attn weights
#   results/fusion_roc.png        — ROC curve + attention weight chart
#   results/fusion_training_curves.png
#   data/model_probs_full.npy     — P1 risk scores for all 590K rows
#
# RUNTIME: ~25 min (Stage 1) + ~15 min (Stage 2) = ~40 min total
#          Stage 1 skipped if model_probs_full.npy already exists.
# ============================================================

In [5]:
%pip install pandas pyarrow numpy==1.26.4 joblib torch scikit-learn matplotlib

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
    --------------------------------------- 0.3/15.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/15.8 MB 2.4 MB/s eta 0:00:07
   --- ------------------------------------ 1.3/15.8 MB 2.5 MB/s eta 0:00:06
   ----- ---------------------------------- 2.1/15.8 MB 2.7 MB/s eta 0:00:06
   ------ --------------------------------- 2.6/15.8 MB 2.9 MB/s eta 0:00:05
   ------- -------------------------------- 3.1/15.8 MB 2.6 MB/s eta 0:00:05
   --------- ------------------------------ 3.9/15.8 MB 2.8 MB/s eta 0:00:05
   ------------ --------------------------- 5.0/15.8 MB 3.1 MB/s eta 0:00:04
   --------------- ------------------------ 6.3/15.8 MB 3.4 MB/s eta 0:00:03
   ------------------- -------------------- 7.6/15.8 MB 3.7 MB/s eta 0:00:03
   ----------------------- ---------------- 9.2/15.8 MB 4.0 MB/s eta 0:00:02
   ------------------------- -------------- 10.0/15.8 MB 4.1 MB/s eta 0:00:02
   ---------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mealpy 3.0.3 requires numpy<=1.26.0,>=1.17.1, but you have numpy 1.26.4 which is incompatible.


In [6]:
# %% Cell 1 — Imports & paths
import os, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, matthews_corrcoef,
                              precision_score, recall_score, f1_score,
                              roc_curve, confusion_matrix)

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_DIR    = os.path.join(ROOT, "data")
MODEL_DIR   = os.path.join(ROOT, "models")
RESULTS_DIR = os.path.join(ROOT, "results")

DEVICE = torch.device("cpu")

print("ROOT:", ROOT)
print("DATA:", DATA_DIR)
print("MODELS:", MODEL_DIR)

TX_PATH    = os.path.join(DATA_DIR, "IEEE CIS", "train_transaction.csv")
SNAP_PATH  = os.path.join(DATA_DIR, "tx_snapshot.parquet")
CTX_PATH   = os.path.join(DATA_DIR, "contextual_features.npy")
P1_PATH    = os.path.join(DATA_DIR, "model_probs_full.npy")

print(f"[NB08] Root  : {ROOT}")
print(f"[NB08] Device: {DEVICE}")

ROOT: f:\rxtj_phase_2
DATA: f:\rxtj_phase_2\data
MODELS: f:\rxtj_phase_2\models
[NB08] Root  : f:\rxtj_phase_2
[NB08] Device: cpu


In [7]:
# %% Cell 2 — Model class definitions
# These are exact copies of the classes in app.py.
# Attribute names must match the saved .pt state_dict keys exactly.
# DO NOT rename attributes — load_state_dict will fail silently or crash.

SEQ_LEN     = 8
CARDINALITY = 4

# ── EARN+ Autoencoder ────────────────────────────────────────────────────────
# State dict key layout (only parametric layers appear):
#   encoder.0=Linear(224,256)  encoder.1=BN(256)
#   [idx 2=ReLU, 3=Dropout — no params]
#   encoder.4=Linear(256,128)  encoder.5=BN(128)
#   [idx 6=ReLU, 7=Dropout — no params]
#   encoder.8=Linear(128,64)   ← NO extra ReLU after this
class FraudAutoencoder(nn.Module):
    def __init__(self, input_dim=224, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, latent_dim)   # index 8 — no ReLU
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, 256),        nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z
    def encode(self, x):
        return self.encoder(x)


# ── EARN+ ResNet extractor ───────────────────────────────────────────────────
# Confirmed architecture from state_dict shape analysis:
#   stem    : Linear(224,128)
#   blocks.0: _ResBlock(128,128)  Identity shortcut
#   blocks.1: _ResBlock(128,64)   Linear shortcut  ← dimension reduction
#   blocks.2: _ResBlock(64,64)    Identity shortcut
#   blocks.3: _ResBlock(64,64)    Identity shortcut
class _ResBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim,  out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(),
            nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim)
        )
        self.shortcut = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.relu     = nn.ReLU()
    def forward(self, x):
        return self.relu(self.block(x) + self.shortcut(x))

class FraudResNet(nn.Module):
    def __init__(self, input_dim=224):
        super().__init__()
        self.stem   = nn.Linear(input_dim, 128)
        self.blocks = nn.Sequential(
            _ResBlock(128, 128),
            _ResBlock(128, 64),
            _ResBlock(64,  64),
            _ResBlock(64,  64),
        )
        self.head = nn.Linear(64, 2)
    def extract(self, x):
        """Return 64-dim EARN+ features (bypass classification head)."""
        return self.blocks(torch.relu(self.stem(x)))
    def forward(self, x):
        return self.head(self.extract(x))


# ── AttentionRXTJ — exact match to app.py attribute names ───────────────────
class _ResNeXtBlock(nn.Module):
    def __init__(self, in_dim, out_dim, cardinality=CARDINALITY):
        super().__init__()
        gd = out_dim // cardinality
        self.paths    = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, gd), nn.BatchNorm1d(gd), nn.ReLU(),
                nn.Linear(gd, gd),     nn.BatchNorm1d(gd)
            ) for _ in range(cardinality)
        ])
        self.shortcut = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.relu     = nn.ReLU()
    def forward(self, x):
        return self.relu(torch.cat([p(x) for p in self.paths], dim=-1) + self.shortcut(x))

class _ResNeXtExtractor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            _ResNeXtBlock(input_dim, 128), _ResNeXtBlock(128, 128),
            _ResNeXtBlock(128, 64),        _ResNeXtBlock(64,  64)
        )
    def forward(self, x):
        return self.net(x)

class _SelfAttentionGRU(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=64, seq_len=SEQ_LEN):
        super().__init__()
        self.seq_len  = seq_len
        self.step_dim = input_dim // seq_len
        self.gru      = nn.GRU(self.step_dim, hidden_dim,
                                num_layers=2, batch_first=True, dropout=0.3)
        self.attention  = nn.Sequential(nn.Linear(hidden_dim, 32), nn.Tanh(),
                                        nn.Linear(32, 1))
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(),
                                        nn.Dropout(0.3), nn.Linear(32, 1),
                                        nn.Identity())
    def forward(self, x):
        x       = x.view(x.size(0), self.seq_len, self.step_dim)
        out, _  = self.gru(x)
        attn_w  = torch.softmax(self.attention(out), dim=1)
        context = (attn_w * out).sum(dim=1)
        return torch.sigmoid(self.classifier(context)).squeeze(1), attn_w

class AttentionRXTJ(nn.Module):
    def __init__(self, input_dim, seq_len=SEQ_LEN):
        super().__init__()
        self.resnext  = _ResNeXtExtractor(input_dim)
        self.attn_gru = _SelfAttentionGRU(input_dim=64, seq_len=seq_len)
        # BUG FIXED: was  self.attn_gru = _SelfSelfAttentionGRU = _SelfAttentionGRU(...)
        # which assigned the class itself to a module-level variable instead of self.attn_gru
    def forward(self, x):
        return self.attn_gru(self.resnext(x))


# ── FusionNet v2 ─────────────────────────────────────────────────────────────
class FusionNet(nn.Module):
    """Attention-weighted MLP: 8 → 64 → 32 → 16 → 1.
    feature_attn is a learnable softmax weight over the 8 inputs.
    At inference: softmax(feature_attn) gives per-feature importance
    used by /account/explain.
    """
    def __init__(self, input_dim=8):
        super().__init__()
        self.feature_attn = nn.Parameter(torch.ones(input_dim))
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        w = torch.softmax(self.feature_attn, dim=0)
        return torch.sigmoid(self.net(x * w)).squeeze(1), w


def _logits(m, x):
    """Raw logits for BCEWithLogitsLoss (no sigmoid — avoids double sigmoid)."""
    return m.net(x * torch.softmax(m.feature_attn, dim=0)).squeeze(1)


# ════════════════════════════════════════════════════════════════════════════
# STAGE 1 — Generate real Phase 1 risk scores using all 224 CSV features
# ════════════════════════════════════════════════════════════════════════════

# Smart resume: skip if valid scores already exist.
# Threshold > 0.05 (not 0.01) — existing bad scores had std=0.011
# which passed the old 0.01 check, causing Stage 1 to be skipped wrongly.
_need_p1 = True
if os.path.exists(P1_PATH):
    _existing = np.load(P1_PATH)
    _snap_len = len(pd.read_parquet(SNAP_PATH, columns=["TransactionID"]))
    _std      = float(np.std(_existing))
    if len(_existing) == _snap_len and _std > 0.05:
        print(f"\n[STAGE 1] Skipping — valid model_probs_full.npy found "
              f"({len(_existing):,} rows, std={_std:.4f})")
        p1_scores = _existing.astype(np.float32)
        _need_p1  = False
    else:
        print(f"\n[STAGE 1] Stale scores detected "
              f"(len={len(_existing)}, std={_std:.4f} ≤ 0.05) — regenerating")

if _need_p1:
    print("\n" + "=" * 60)
    print("  STAGE 1: Generating Phase 1 risk scores (all 224 features)")
    print("  Pipeline: CSV(224) → AE(64)+ResNet(64) → Nystroem → IPCA(50)")
    print("=" * 60)

    # ── Load preprocessors ────────────────────────────────────────────────────
    print("\n[STAGE 1] Loading Phase 1 artifacts...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        imputer  = joblib.load(os.path.join(MODEL_DIR, "imputer.pkl"))
        scaler   = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
        nystroem = joblib.load(os.path.join(MODEL_DIR, "nystroem.pkl"))
        ipca     = joblib.load(os.path.join(MODEL_DIR, "incremental_pca.pkl"))
        ifm      = joblib.load(os.path.join(MODEL_DIR, "isolation_forest.pkl"))

    NYS_DIM  = int(nystroem.n_features_in_)   # 128
    IPCA_DIM = int(ipca.n_components_)          # 50
    print(f"  nystroem expects: {NYS_DIM} features  (AE 64 + ResNet 64)")
    print(f"  ipca components : {IPCA_DIM}")

    # ── Load EARN+ models ────────────────────────────────────────────────────
    ae_state = torch.load(os.path.join(MODEL_DIR, "autoencoder.pt"), map_location="cpu")
    ae_in    = ae_state["encoder.0.weight"].shape[1]
    ae_model = FraudAutoencoder(input_dim=ae_in)
    ae_model.load_state_dict(ae_state)          # strict=True — exact match
    ae_model.eval()
    print(f"  FraudAutoencoder loaded (input={ae_in}) ✓")

    rn_model = FraudResNet(input_dim=ae_in)
    rn_state = torch.load(os.path.join(MODEL_DIR, "resnet_extractor.pt"), map_location="cpu")
    rn_model.load_state_dict(rn_state, strict=False)   # strict=False — stem key tolerance
    rn_model.eval()
    print(f"  FraudResNet loaded (extract→64) ✓")

    p1_model = AttentionRXTJ(input_dim=IPCA_DIM).to(DEVICE)
    p1_model.load_state_dict(
        torch.load(os.path.join(MODEL_DIR, "attention_rxtj.pt"), map_location=DEVICE))
    p1_model.eval()
    print(f"  AttentionRXTJ loaded (input={IPCA_DIM}) ✓")

    cfg_p1    = json.load(open(os.path.join(RESULTS_DIR, "deployment_config.json")))
    W_MODEL   = float(cfg_p1["W_MODEL"])
    W_IFM     = float(cfg_p1["W_IFM"])
    THRESHOLD = float(cfg_p1["THRESHOLD"])
    print(f"  W_MODEL={W_MODEL:.4f}  W_IFM={W_IFM:.4f}  THRESHOLD={THRESHOLD:.4f}")

    # ── Load ALL 224 named columns from original CSV ──────────────────────────
    # BUG FIXED (vs old NB08): was building X_raw with only 4 columns filled,
    # leaving 220 as NaN → imputed to means → all rows identical → std=0.011.
    # NOW: uses imputer.feature_names_in_ to load the exact 224 columns the
    # imputer was trained on, with categorical encoding matching NB01.
    print(f"\n[STAGE 1] Loading all 224 named features from train_transaction.csv...")
    t0           = time.time()
    feature_cols = list(imputer.feature_names_in_)
    load_cols    = ["TransactionID"] + [c for c in feature_cols if c != "TransactionID"]
    df_full      = pd.read_csv(TX_PATH, usecols=load_cols)
    print(f"  CSV loaded: {len(df_full):,} rows · {len(df_full.columns)} cols "
          f"({time.time()-t0:.1f}s)")

    # Encode string/categorical columns (matches NB01 label encoding via category codes)
    str_cols = []
    for col in feature_cols:
        if col in df_full.columns and df_full[col].dtype == object:
            df_full[col] = df_full[col].astype("category").cat.codes.astype(np.float32)
            df_full[col] = df_full[col].replace(-1, np.nan)
            str_cols.append(col)
    if str_cols:
        print(f"  Encoded {len(str_cols)} categorical cols: {str_cols}")

    # Align to snapshot row order (sorted by card1, TransactionDT in NB06)
    snap_ids = pd.read_parquet(SNAP_PATH, columns=["TransactionID"])
    df_full  = snap_ids.merge(df_full, on="TransactionID", how="left")
    print(f"  Aligned to snapshot: {len(df_full):,} rows "
          f"{'✓' if len(df_full) == len(snap_ids) else '✗ MISMATCH'}")

    X_raw   = df_full[feature_cols].values.astype(np.float32)
    nan_pct = 100.0 * np.isnan(X_raw).mean()
    print(f"  X_raw: {X_raw.shape}  NaN%: {nan_pct:.1f}%  "
          f"(was 97% with 4-col approach — should be much lower now)")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        X_sc = scaler.transform(imputer.transform(X_raw)).astype(np.float32)
    print(f"  Imputed+scaled: {X_sc.shape}")

    # ── EARN+ feature extraction: AE(64) + ResNet(64) → concat(128) ──────────
    print(f"\n[STAGE 1] Extracting EARN+ features (~5 min)...")
    BATCH     = 4096
    Z_ae_list = []
    Z_rn_list = []
    t0 = time.time()
    with torch.no_grad():
        for s in range(0, len(X_sc), BATCH):
            # BUG FIXED: was X_sc[s+BATCH] (no colon) → IndexError
            xb = torch.FloatTensor(X_sc[s:s+BATCH])
            Z_ae_list.append(ae_model.encode(xb).cpu().numpy())
            Z_rn_list.append(rn_model.extract(xb).cpu().numpy())
            if (s // BATCH) % 30 == 0:
                print(f"  {min(s+BATCH, len(X_sc)):>7,}/{len(X_sc):,}  ({time.time()-t0:.0f}s)")

    Z_earn = np.hstack([
        np.vstack(Z_ae_list).astype(np.float32),
        np.vstack(Z_rn_list).astype(np.float32),
    ])
    print(f"  Z_earn: {Z_earn.shape}  ← must be (N, {NYS_DIM})")
    assert Z_earn.shape[1] == NYS_DIM, \
        f"EARN dim {Z_earn.shape[1]} ≠ Nystroem expects {NYS_DIM}"
    print(f"  ✓ Dimension check passed")

    # ── Nystroem + IPCA ───────────────────────────────────────────────────────
    print(f"\n[STAGE 1] Nystroem + IPCA in chunks (~8 min)...")
    CHUNK  = 8000
    X_ipca = np.zeros((len(Z_earn), IPCA_DIM), dtype=np.float32)
    t0     = time.time()
    for s in range(0, len(Z_earn), CHUNK):
        # BUG FIXED: was Z_earn[s] (single row) → shape error
        e = min(s + CHUNK, len(Z_earn))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            X_ipca[s:e] = ipca.transform(
                nystroem.transform(Z_earn[s:e])
            ).astype(np.float32)
        if s % 80000 == 0:
            print(f"  {e:>7,}/{len(Z_earn):,}  ({time.time()-t0:.0f}s)")
    print(f"  Done in {time.time()-t0:.0f}s")

    # ── Ensemble scoring ──────────────────────────────────────────────────────
    print(f"\n[STAGE 1] Ensemble scoring (AttentionRXTJ + IFM)...")
    all_risk = []
    with torch.no_grad():
        for s in range(0, len(X_ipca), BATCH):
            # BUG FIXED: was X_ipca[s] (single row) → shape error
            e        = min(s + BATCH, len(X_ipca))
            probs, _ = p1_model(torch.FloatTensor(X_ipca[s:e]).to(DEVICE))
            pnp      = probs.cpu().numpy()
            ifm_norm = 1.0 / (1.0 + np.exp(ifm.decision_function(X_ipca[s:e])))
            all_risk.extend((W_MODEL * pnp + W_IFM * ifm_norm).tolist())
            if (s // BATCH) % 50 == 0:
                print(f"  {e:>7,}/{len(X_ipca):,}")

    p1_scores = np.array(all_risk, dtype=np.float32)
    print(f"\n  ┌──────────────────────────────────────────────┐")
    print(f"  │  P1 scores — full 224-feature pipeline        │")
    print(f"  │  min    : {p1_scores.min():.4f}                          │")
    print(f"  │  mean   : {p1_scores.mean():.4f}                          │")
    print(f"  │  max    : {p1_scores.max():.4f}                          │")
    print(f"  │  std    : {p1_scores.std():.4f}  ← must be > 0.05       │")
    print(f"  │  fraud% : {(p1_scores>=THRESHOLD).mean()*100:.2f}%  ← must be ~3–10%    │")
    print(f"  └──────────────────────────────────────────────┘")

    if p1_scores.std() < 0.02:
        print("\n  WARNING: std still low — ResNet stem may not have loaded.")
        print("  Other 7 features will carry the model. Proceeding.")
    else:
        print(f"\n  ✓ P1 scores have real spread — feature is useful")

    np.save(P1_PATH, p1_scores)
    X_ctx       = np.load(CTX_PATH)
    X_ctx[:, 7] = p1_scores
    np.save(CTX_PATH, X_ctx)
    print(f"  model_probs_full.npy saved")
    print(f"  contextual_features.npy col-7 patched "
          f"({X_ctx[:,7].min():.4f}–{X_ctx[:,7].max():.4f})")


# ════════════════════════════════════════════════════════════════════════════
# STAGE 2 — Train FusionNet v2
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  STAGE 2: Training FusionNet v2")
print("=" * 60)


[STAGE 1] Stale scores detected (len=590540, std=0.0110 ≤ 0.05) — regenerating

  STAGE 1: Generating Phase 1 risk scores (all 224 features)
  Pipeline: CSV(224) → AE(64)+ResNet(64) → Nystroem → IPCA(50)

[STAGE 1] Loading Phase 1 artifacts...
  nystroem expects: 128 features  (AE 64 + ResNet 64)
  ipca components : 50
  FraudAutoencoder loaded (input=224) ✓
  FraudResNet loaded (extract→64) ✓
  AttentionRXTJ loaded (input=50) ✓
  W_MODEL=0.5343  W_IFM=0.4657  THRESHOLD=0.5000

[STAGE 1] Loading all 224 named features from train_transaction.csv...
  CSV loaded: 590,540 rows · 225 cols (13.1s)
  Encoded 13 categorical cols: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
  Aligned to snapshot: 590,540 rows ✓
  X_raw: (590540, 224)  NaN%: 12.2%  (was 97% with 4-col approach — should be much lower now)
  Imputed+scaled: (590540, 224)

[STAGE 1] Extracting EARN+ features (~5 min)...
    4,096/590,540  (0s)
  126,976/590,540  (1s)
  24

In [8]:
# %% Cell 3 — Load features + labels
print("\n[STAGE 2] Loading contextual features...")
X        = np.load(CTX_PATH)
y        = np.load(os.path.join(DATA_DIR, "fusion_labels.npy"))
acct_ids = np.load(os.path.join(DATA_DIR, "fusion_account_ids.npy"))
with open(os.path.join(DATA_DIR, "fusion_feature_names.json")) as f:
    FEATURE_NAMES = json.load(f)

print(f"  X={X.shape}  positives={int(y.sum()):,} ({100*y.mean():.2f}%)")
print(f"  P1 col: {X[:,7].min():.4f}–{X[:,7].max():.4f}  std={X[:,7].std():.4f}")
print(f"  drift : {X[:,6].min():.4f}–{X[:,6].max():.4f}  "
      f"corr={float(np.corrcoef(X[:,6], y)[0,1]):.4f}")

X = np.nan_to_num(X, nan=0.0)


[STAGE 2] Loading contextual features...
  X=(590540, 8)  positives=20,663 (3.50%)
  P1 col: 0.2887–0.7449  std=0.0114
  drift : 0.0000–1.0000  corr=0.1517


In [9]:
# %% Cell 4 — Scale
fusion_scaler = StandardScaler()
X_sc          = fusion_scaler.fit_transform(X).astype(np.float32)
joblib.dump(fusion_scaler, os.path.join(MODEL_DIR, "fusion_scaler.pkl"))
print("\n  fusion_scaler.pkl saved")


  fusion_scaler.pkl saved


In [10]:
# %% Cell 5 — Account-stratified split (prevents data leakage across account boundary)
print("\n[STAGE 2] Account-stratified split (70/15/15)...")
unique_a = np.unique(acct_ids)
fa       = set(acct_ids[y == 1])
al       = np.array([1 if a in fa else 0 for a in unique_a])
a_tr, a_tmp, _, _ = train_test_split(unique_a, al,
                                      test_size=0.30, stratify=al, random_state=42)
a_va, a_te, _, _  = train_test_split(
    a_tmp,
    np.array([1 if a in fa else 0 for a in a_tmp]),
    test_size=0.50, random_state=42)

X_tr = X_sc[np.isin(acct_ids, a_tr)]; y_tr = y[np.isin(acct_ids, a_tr)]
X_va = X_sc[np.isin(acct_ids, a_va)]; y_va = y[np.isin(acct_ids, a_va)]
X_te = X_sc[np.isin(acct_ids, a_te)]; y_te = y[np.isin(acct_ids, a_te)]
print(f"  Train={len(X_tr):,} ({y_tr.mean()*100:.2f}%)  "
      f"Val={len(X_va):,} ({y_va.mean()*100:.2f}%)  "
      f"Test={len(X_te):,} ({y_te.mean()*100:.2f}%)")


[STAGE 2] Account-stratified split (70/15/15)...
  Train=413,219 (3.68%)  Val=99,138 (3.02%)  Test=78,183 (3.13%)


In [11]:
# %% Cell 6 — Model + optimiser setup
INPUT_DIM  = X_tr.shape[1]
model_f    = FusionNet(INPUT_DIM).to(DEVICE)
pos_weight = torch.tensor([(y_tr==0).sum() / (y_tr==1).sum()]).float()
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
optimizer  = torch.optim.Adam(model_f.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=200, eta_min=1e-5)

BATCH_SIZE = 4096
EPOCHS     = 200
PATIENCE   = 25

print(f"\n[STAGE 2] FusionNet v2  input={INPUT_DIM}  pos_weight={pos_weight.item():.2f}")
print(f"  Architecture: {INPUT_DIM}→64→32→16→1  +  feature_attn[{INPUT_DIM}]")

loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr)),
    batch_size=BATCH_SIZE, shuffle=True)
Xv_t = torch.FloatTensor(X_va).to(DEVICE)


[STAGE 2] FusionNet v2  input=8  pos_weight=26.15
  Architecture: 8→64→32→16→1  +  feature_attn[8]


In [12]:
# %% Cell 7 — Training loop (monitors AUC, not loss)
print(f"\n[STAGE 2] Training (target AUC ≥ 0.85)...")
best_auc, best_state, no_imp = 0.0, None, 0
train_losses, val_aucs       = [], []
t_tr = time.time()

for ep in range(1, EPOCHS + 1):
    model_f.train()
    ep_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        # BUG FIXED: was criterion(logits(...)) — logits() is undefined.
        # Correct name is _logits() (defined above).
        loss = criterion(_logits(model_f, xb), yb)
        loss.backward()
        # BUG FIXED: was clip_grad_norm() (missing underscore) → AttributeError
        nn.utils.clip_grad_norm_(model_f.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item() * len(xb)
    scheduler.step()

    model_f.eval()
    with torch.no_grad():
        vp = torch.sigmoid(_logits(model_f, Xv_t)).cpu().numpy()
    vauc = roc_auc_score(y_va, vp)

    train_losses.append(ep_loss / len(X_tr))
    val_aucs.append(vauc)

    if ep % 10 == 0 or ep <= 5:
        print(f"  Epoch {ep:>3}  loss={ep_loss/len(X_tr):.4f}  "
              f"AUC={vauc:.4f}  lr={optimizer.param_groups[0]['lr']:.6f}")

    if vauc > best_auc:
        best_auc   = vauc; no_imp = 0
        best_state = {k: v.clone() for k, v in model_f.state_dict().items()}
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f"  Early stopping at epoch {ep} (patience={PATIENCE})")
            break

model_f.load_state_dict(best_state)
model_f.eval()
print(f"\n  Best val AUC: {best_auc:.4f}  ({time.time()-t_tr:.0f}s)")


[STAGE 2] Training (target AUC ≥ 0.85)...
  Epoch   1  loss=1.3216  AUC=0.6638  lr=0.000500
  Epoch   2  loss=1.2293  AUC=0.6844  lr=0.000500
  Epoch   3  loss=1.2033  AUC=0.6944  lr=0.000500
  Epoch   4  loss=1.1974  AUC=0.6992  lr=0.000500
  Epoch   5  loss=1.1951  AUC=0.7024  lr=0.000499
  Epoch  10  loss=1.1863  AUC=0.7073  lr=0.000497
  Epoch  20  loss=1.1759  AUC=0.7128  lr=0.000488
  Epoch  30  loss=1.1697  AUC=0.7154  lr=0.000473
  Epoch  40  loss=1.1644  AUC=0.7166  lr=0.000453
  Epoch  50  loss=1.1614  AUC=0.7168  lr=0.000428
  Epoch  60  loss=1.1560  AUC=0.7180  lr=0.000399
  Epoch  70  loss=1.1541  AUC=0.7182  lr=0.000366
  Epoch  80  loss=1.1510  AUC=0.7192  lr=0.000331
  Epoch  90  loss=1.1505  AUC=0.7201  lr=0.000293
  Epoch 100  loss=1.1483  AUC=0.7206  lr=0.000255
  Epoch 110  loss=1.1496  AUC=0.7209  lr=0.000217
  Epoch 120  loss=1.1474  AUC=0.7202  lr=0.000179
  Epoch 130  loss=1.1478  AUC=0.7209  lr=0.000144
  Epoch 140  loss=1.1465  AUC=0.7219  lr=0.000111
  Epoch

In [13]:
# %% Cell 8 — Jaya threshold optimisation
print("\n[STAGE 2] Jaya threshold optimisation...")
with torch.no_grad():
    vp_np = torch.sigmoid(_logits(model_f, Xv_t)).cpu().numpy()

def _cost(t, p, l):
    pred = (p >= t).astype(int)
    fp   = ((pred==1) & (l==0)).sum();  fn = ((pred==0) & (l==1)).sum()
    tp   = ((pred==1) & (l==1)).sum();  tn = ((pred==0) & (l==0)).sum()
    return 2.0*fp/(fp+tn+1e-9) + fn/(fn+tp+1e-9)

pop = np.random.uniform(0.1, 0.9, 40)
c   = np.array([_cost(t, vp_np, y_va) for t in pop])
for _ in range(150):
    bi, wi = np.argmin(c), np.argmax(c)
    r1, r2 = np.random.rand(40), np.random.rand(40)
    np2    = np.clip(pop + r1*(pop[bi]-np.abs(pop)) - r2*(pop[wi]-np.abs(pop)), 0.05, 0.95)
    nc     = np.array([_cost(t, vp_np, y_va) for t in np2])
    m      = nc < c; pop = np.where(m, np2, pop); c = np.where(m, nc, c)

OPT_T  = float(pop[np.argmin(c)])
HIGH_T = min(OPT_T + 0.15, 0.90)
ELEV_T = max(OPT_T - 0.10, 0.25)
print(f"  Optimal  : {OPT_T:.4f}")
print(f"  HIGH     : {HIGH_T:.4f}")
print(f"  ELEVATED : {ELEV_T:.4f}")
print(f"  Val cost : {_cost(OPT_T, vp_np, y_va):.4f}")


[STAGE 2] Jaya threshold optimisation...
  Optimal  : 0.5389
  HIGH     : 0.6889
  ELEVATED : 0.4389
  Val cost : 0.8540


In [14]:
# %% Cell 9 — Test set evaluation
print("\n[STAGE 2] Test set evaluation...")
Xt_t = torch.FloatTensor(X_te).to(DEVICE)
with torch.no_grad():
    test_probs = torch.sigmoid(_logits(model_f, Xt_t)).cpu().numpy()
    attn_final = torch.softmax(model_f.feature_attn, dim=0).cpu().numpy()

test_preds = (test_probs >= OPT_T).astype(int)
auc  = roc_auc_score(y_te, test_probs)
mcc  = matthews_corrcoef(y_te, test_preds)
prec = precision_score(y_te, test_preds, zero_division=0)
rec  = recall_score(y_te, test_preds, zero_division=0)
f1   = f1_score(y_te, test_preds, zero_division=0)
cm   = confusion_matrix(y_te, test_preds)
TP, FP = int(cm[1,1]), int(cm[0,1])
FN, TN = int(cm[1,0]), int(cm[0,0])

print(f"\n  ┌─────────────────────────────────────────┐")
print(f"  │  AUC-ROC   : {auc:.4f}                     │")
print(f"  │  MCC       : {mcc:.4f}                     │")
print(f"  │  Precision : {prec:.4f}                     │")
print(f"  │  Recall    : {rec:.4f}                     │")
print(f"  │  F1        : {f1:.4f}                     │")
print(f"  │  TP={TP:<5}  FP={FP:<5}  TN={TN:<6}  FN={FN:<4} │")
print(f"  └─────────────────────────────────────────┘")

if auc >= 0.85:
    print(f"\n  ✓ AUC target met ({auc:.4f} ≥ 0.85)")
else:
    print(f"\n  ⚠  AUC={auc:.4f} (< 0.85)")

print(f"\n  Learned attention weights (feature importance):")
for i, name in enumerate(FEATURE_NAMES):
    bar = "█" * int(attn_final[i] * 60)
    print(f"    {name:<25}  {attn_final[i]:.4f}  {bar}")


[STAGE 2] Test set evaluation...

  ┌─────────────────────────────────────────┐
  │  AUC-ROC   : 0.7537                     │
  │  MCC       : 0.1708                     │
  │  Precision : 0.0978                     │
  │  Recall    : 0.5202                     │
  │  F1        : 0.1647                     │
  │  TP=1272   FP=11733  TN=64005   FN=1173 │
  └─────────────────────────────────────────┘

  ⚠  AUC=0.7537 (< 0.85)

  Learned attention weights (feature importance):
    amount_z_score             0.1577  █████████
    merchant_novelty           0.1398  ████████
    geo_displacement           0.0853  █████
    hour_deviation             0.1174  ███████
    device_novelty             0.1077  ██████
    velocity_ratio             0.1110  ██████
    behavioral_drift           0.1988  ███████████
    p1_risk_score              0.0824  ████


In [15]:
# %% Cell 10 — ROC + training curve plots
fpr_a, tpr_a, _ = roc_curve(y_te, test_probs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr_a, tpr_a, color="#0099cc", lw=2, label=f"FusionNet v2 AUC={auc:.4f}")
axes[0].plot([0,1],[0,1],"k--",lw=0.8)
axes[0].axvline(x=0.15, color="gray", ls=":", alpha=0.5, label="15% FPR target")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("FusionNet v2 ROC Curve"); axes[0].legend()
axes[1].bar(FEATURE_NAMES, attn_final, color="#00d4ff", edgecolor="#0099cc")
axes[1].set_title("Feature Attention Weights")
axes[1].set_ylabel("Weight"); axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fusion_roc.png"), dpi=150, bbox_inches="tight")
plt.close()

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(train_losses, label="Train loss", color="#0099cc")
ax2.plot(val_aucs,     label="Val AUC",   color="#00e5a0")
ax2.set_xlabel("Epoch"); ax2.set_title("FusionNet v2 Training Curves"); ax2.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fusion_training_curves.png"),
            dpi=150, bbox_inches="tight")
plt.close()

In [16]:
# %% Cell 11 — Save model + config
torch.save(model_f.state_dict(), os.path.join(MODEL_DIR, "fusion_net.pt"))

fusion_config = {
    "input_dim":            INPUT_DIM,
    "fusion_threshold":     round(OPT_T, 6),
    "high_threshold":       round(HIGH_T, 6),
    "elevated_threshold":   round(ELEV_T, 6),
    "fusion_auc":           round(float(auc), 6),
    "fusion_mcc":           round(float(mcc), 6),
    "fusion_precision":     round(float(prec), 6),
    "fusion_recall":        round(float(rec), 6),
    "fusion_f1":            round(float(f1), 6),
    "true_positives":       TP,
    "false_positives":      FP,
    "true_negatives":       TN,
    "false_negatives":      FN,
    "feature_names":        FEATURE_NAMES,
    "optimal_attn_weights": [round(float(w), 6) for w in attn_final],
    "model_version":        "fusionnet_v2",
    "training_epochs":      len(train_losses),
    "best_val_auc":         round(float(best_auc), 6),
    "pos_weight_used":      round(float(pos_weight.item()), 4),
    "p1_full_features":     True,    # marks that all 224 CSV features were used
}
with open(os.path.join(RESULTS_DIR, "fusion_config.json"), "w") as f:
    json.dump(fusion_config, f, indent=2)

print(f"\n[NB08] Saved:")
print(f"  models/fusion_net.pt          ✓")
print(f"  models/fusion_scaler.pkl      ✓")
print(f"  results/fusion_config.json    ✓")
print(f"  results/fusion_roc.png        ✓")
print(f"  results/fusion_training_curves.png  ✓")
print(f"\n[NB08] ✓ Complete.  AUC={auc:.4f}  MCC={mcc:.4f}  threshold={OPT_T:.4f}")
print(f"\n[NB08] NEXT STEP → merge app_phase2_endpoints.py into app.py")


[NB08] Saved:
  models/fusion_net.pt          ✓
  models/fusion_scaler.pkl      ✓
  results/fusion_config.json    ✓
  results/fusion_roc.png        ✓
  results/fusion_training_curves.png  ✓

[NB08] ✓ Complete.  AUC=0.7537  MCC=0.1708  threshold=0.5389

[NB08] NEXT STEP → merge app_phase2_endpoints.py into app.py
